In [16]:
import pandas as pd

# Read the CSV file as a dataframe
df = pd.read_csv(r'Z:\HTOC\Data_Analytics\Data\Observed_Tags\htoc_observed_indicator_tags.csv')

# Filter to keep only records where threat_category is 'threat actor'
df = df[df['threat_category'] == 'THREAT ACTOR']

# Read the Excel file as a dataframe
df_scores = pd.read_excel(r'Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx')

# Match indicators from threat scores with their threat actors
# Only keep indicators that are in df_scores
df_merged = df_scores.merge(
    df[['indicator', 'tag']],  # Select only indicator and tag (threat actor) columns
    left_on='Indicator',
    right_on='indicator',
    how='left'  # Keep all indicators from df_scores, even if no threat actor match
)

# Rename the tag column to be more descriptive
df_merged = df_merged.rename(columns={'tag': 'Threat Actor'})

# Remove the duplicate indicator column (keep 'Indicator' from df_scores, drop 'indicator' from df)
df_merged = df_merged.drop(columns=['indicator'])

# Aggregate multiple threat actors per indicator into a single row
# Group only by 'Indicator' to ensure one row per indicator
# Take first value for all other columns (they should be the same per indicator)
other_cols = [col for col in df_merged.columns if col not in ['Indicator', 'Threat Actor']]
agg_dict = {col: 'first' for col in other_cols}
agg_dict['Threat Actor'] = lambda x: ', '.join(x.dropna().unique()) if x.notna().any() else None

df_merged = df_merged.groupby('Indicator', as_index=False).agg(agg_dict)

# Convert empty strings to NaN
df_merged['Threat Actor'] = df_merged['Threat Actor'].replace('', None)

# Verify all indicators from df_scores are included
original_count = len(df_scores)
merged_count = len(df_merged)
print(f"Original indicators in df_scores: {original_count}")
print(f"Indicators in merged dataframe: {merged_count}")
print(f"All indicators preserved: {original_count == merged_count}")

# Display information about the merged dataframe
print(f"\nMerged dataframe shape: {df_merged.shape}")
print(f"\nColumns: {df_merged.columns.tolist()}")
print(f"\nNumber of indicators with threat actors: {df_merged['Threat Actor'].notna().sum()}")
print(f"\nNumber of indicators without threat actors: {df_merged['Threat Actor'].isna().sum()}")
print(f"\nFirst few rows:")
df_merged


Original indicators in df_scores: 1416
Indicators in merged dataframe: 1415
All indicators preserved: False

Merged dataframe shape: (1415, 14)

Columns: ['Indicator', 'Last Observed', 'Indicator Type', 'VirusTotal Malicious Score', 'Observation Yearly Count', 'ThreatConnect Rating', 'Observation Penalty Multiplier', 'Botnet Flag', 'False Positives', 'Partners', 'Threat Score', 'Severity', 'Score Explanation', 'Threat Actor']

Number of indicators with threat actors: 426

Number of indicators without threat actors: 989

First few rows:


,Indicator,Last Observed,Indicator Type,VirusTotal Malicious Score,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,Threat Score,Severity,Score Explanation,Threat Actor
0,1-you.njalla.no,2025-11-13,Host,0,17.0,5,0.999068,0,0,"VA, NIH",167,low,Severity: low. Top drivers: TC confidence; TC ...,Wicked Panda
1,101.71.130.99,2025-11-17,Address,10,33.0,3,0.998192,0,0,"DHA, OS, FDA, CMS, NIH, HHS",427,medium,Severity: medium. Top drivers: VT malicious (l...,None
2,101.89.174.236,2025-11-12,Address,9,176.0,3,0.990356,0,0,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS",374,medium,Severity: medium. Top drivers: VT malicious (l...,None
3,102.164.252.150,2025-10-29,Address,1,4.0,1,0.999781,1,0,"CMS, VA, NIH",199,low,Severity: low. Top drivers: TOR activity; TC c...,Mr Hamza Group
4,102.209.18.96,2025-10-24,Address,1,9.0,1,0.999507,1,0,"FDA, VA",199,low,Severity: low. Top drivers: TOR activity; TC c...,Mr Hamza Group
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1410,www.prosinthecity.com,2025-11-12,Host,0,15.0,4,0.999178,0,0,"DHA, OS, CMS, NIH",138,low,Severity: low. Top drivers: TC confidence; Con...,Mustard Tempest
1411,www.shorturl.at/,2025-11-14,Stripped URL,0,173.0,3,0.990521,0,0,"FDA, CDC",199,low,Severity: low. Top drivers: TC confidence; Con...,None
1412,www.sthda.com,2025-11-16,Host,0,250.0,4,0.986301,0,0,"DHA, OS, FDA, CMS, VA, HRSA, NIH, IHS",108,low,Severity: low. Top drivers: TC confidence; Con...,None
1413,yourpensionmeeting.com,2025-11-17,Host,0,125.0,3,0.993151,0,0,"DHA, CMS, VA, NIH",168,low,Severity: low. Top drivers: TC confidence; Con...,None
